# Saving class probabilities

By default, a segmentation model writes one class per pixel: the class with the highest probability. With `save_probabilities=True`, the class probabilities (after softmax) are also written to a separate GeoTIFF, with one band per class. The class prediction itself stays the same.

The probabilities tell you how sure the model is about each pixel. You can use them to find uncertain areas, leave uncertain pixels unclassified, choose your own decision rule, or combine the predictions of several models.

In [ ]:
from pathlib import Path

from lightning import seed_everything
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint
from matplotlib import pyplot as plt

from eotorch.data import (
    RasterDataModule,
    SegmentationTask,
    get_segmentation_dataset,
    splits,
)
from eotorch.processing.normalize import band_statistics

# class mapping, used for plotting and for the band names of the probabilities file
class_mapping = {
    1: "Optically deep water",
    2: "Sand",
    3: "Coral",
    4: "SAV",
}
DATASET_DIR = Path("tests/data")

### Train a model

We quickly train a model on the test data. The steps are explained in detail in [segmentation.ipynb](segmentation.ipynb).

If you already have a checkpoint of a segmentation model, set `checkpoint` to its path and skip this section.

In [ ]:
seed_everything(43, workers=False)

ds = get_segmentation_dataset(
    images_dir=DATASET_DIR,
    labels_dir=DATASET_DIR,
    image_glob="image_*.tif",
    label_glob="label_*.tif",
    all_image_bands=("red", "green", "blue"),
    rgb_bands=("red", "green", "blue"),
    class_mapping=class_mapping,
    reduce_zero_label=True,
)
train_ds, val_ds = splits.file_wise_split(
    dataset=ds, val_img_files=DATASET_DIR / "image_3.tif"
)
mean, std = band_statistics(train_ds)
module = RasterDataModule(
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=16,
    patch_size=64,
    mean=mean,
    std=std,
)
task = SegmentationTask(
    model="deepresunet",
    in_channels=3,
    num_classes=len(class_mapping),
    ignore_index=-1,  # because reduce_zero_label=True
)
trainer = Trainer(
    callbacks=[ModelCheckpoint(monitor="val/F1_Score", mode="max")],
    max_epochs=30,
)
trainer.fit(model=task, datamodule=module)
checkpoint = trainer.checkpoint_callback.best_model_path

### Predict classes and probabilities

With `save_probabilities=True`, two files are written:

- `image_3_pred.tif`: the predicted class of each pixel, as without the option
- `image_3_pred_probabilities.tif`: the probability of each class, as float32, one band per class

The bands are named after the classes in the class mapping saved in the checkpoint. Since the model was trained with `reduce_zero_label=True`, band 1 holds the probability of class 1, band 2 of class 2, and so on. Pixels without a prediction are NaN.

Saving the probabilities works with and without `blend`. With `blend=True`, the probabilities of overlapping patches are averaged, and the classes are derived from the averaged probabilities.

In [ ]:
pred_path = SegmentationTask.predict_on_tif_file(
    tif_file_path=DATASET_DIR / "image_3.tif",
    checkpoint_path=checkpoint,
    out_file_path="predictions/image_3_pred.tif",
    blend=True,
    save_probabilities=True,
)
prob_path = pred_path.with_name(f"{pred_path.stem}_probabilities.tif")
print(pred_path)
print(prob_path)